
# AI & Machine Learning Research Paper Analytics
## End-to-End arXiv Research Intelligence & Predictive Modeling

This notebook provides a professional-grade analysis pipeline for AI/ML research papers from arXiv.

## Included Sections
- Full Exploratory Data Analysis (EDA)
- Data Cleaning & Preprocessing
- Missing Value Analysis
- Outlier Detection
- Feature Engineering
- Research Trend Analysis
- Topic & Category Insights
- Correlation Analysis
- Author & Publication Analysis
- Advanced Visualizations
- Predictive Machine Learning Models
- Feature Importance
- Business & Research Insights

---

## Dataset Overview
- Rows: **7,701**
- Columns: **26**



In [ ]:

# =========================
# IMPORT LIBRARIES
# =========================

import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
from sklearn.feature_extraction.text import TfidfVectorizer

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Settings
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

print("Libraries loaded successfully.")


In [ ]:

# =========================
# LOAD DATASET
# =========================

df = pd.read_csv(r"/mnt/data/arxiv_ai_ml_papers.csv")

print("Dataset Shape:", df.shape)

df.head()


## Dataset Inspection

In [ ]:

df.info()


In [ ]:

df.describe(include='all').T


## Missing Value Analysis

In [ ]:

missing = df.isnull().sum().sort_values(ascending=False)

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing Values': missing.values,
    'Missing Percentage': (missing.values / len(df)) * 100
})

missing_df.head(20)


In [ ]:

plt.figure(figsize=(14,6))

sns.barplot(
    x=missing_df['Column'][:20],
    y=missing_df['Missing Percentage'][:20]
)

plt.xticks(rotation=90)

plt.title("Top Missing Value Percentages")

plt.show()


## Data Cleaning & Preprocessing

In [ ]:

# Remove duplicates

duplicates = df.duplicated().sum()

print("Duplicate Rows:", duplicates)

df = df.drop_duplicates()

print("Shape After Deduplication:", df.shape)


In [ ]:

# Convert numeric columns automatically

for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except:
        pass

print("Automatic numeric conversion completed.")


## Exploratory Data Analysis

In [ ]:

# Numerical Columns

numeric_cols = df.select_dtypes(include='number').columns.tolist()

print(numeric_cols)


In [ ]:

# Distribution Plots

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:6]:

    plt.figure(figsize=(10,5))

    sns.histplot(df[col].dropna(), kde=True)

    plt.title(f"Distribution of {col}")

    plt.show()


## Correlation Analysis

In [ ]:

numeric_df = df.select_dtypes(include='number')

corr = numeric_df.corr()

plt.figure(figsize=(14,10))

sns.heatmap(
    corr,
    cmap='coolwarm',
    annot=False
)

plt.title("Correlation Matrix")

plt.show()


## Outlier Detection

In [ ]:

# Boxplots

numeric_cols = df.select_dtypes(include='number').columns.tolist()

for col in numeric_cols[:5]:

    plt.figure(figsize=(12,4))

    sns.boxplot(x=df[col])

    plt.title(f"Outlier Detection - {col}")

    plt.show()


In [ ]:

# IQR-Based Outlier Detection

if len(numeric_cols) > 0:

    col = numeric_cols[0]

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[
        (df[col] < lower) |
        (df[col] > upper)
    ]

    print(f"Outliers in {col}: {len(outliers)}")


## Research Trend Analysis

In [ ]:

# Time-Based Research Trends

time_cols = [
    c for c in df.columns
    if 'year' in c.lower() or 'date' in c.lower()
]

print("Potential Time Columns:", time_cols)

if len(time_cols) > 0:

    col = time_cols[0]

    try:

        trend = df[col].value_counts().sort_index()

        plt.figure(figsize=(14,5))

        trend.plot()

        plt.title(f"Research Trend Over Time - {col}")

        plt.show()

    except:
        print("Trend analysis skipped.")


## Text & NLP Feature Engineering

In [ ]:

# Text Columns

text_cols = df.select_dtypes(include='object').columns.tolist()

print(text_cols)


In [ ]:

# Feature Engineering

# Missing values count
df['missing_feature_count'] = df.isnull().sum(axis=1)

# Text length features
text_cols = df.select_dtypes(include='object').columns.tolist()

for col in text_cols[:3]:
    df[f'{col}_length'] = df[col].astype(str).apply(len)

df.head()


In [ ]:

# TF-IDF Example on Text Data

text_cols = df.select_dtypes(include='object').columns.tolist()

if len(text_cols) > 0:

    sample_col = text_cols[0]

    vectorizer = TfidfVectorizer(
        max_features=20,
        stop_words='english'
    )

    text_data = df[sample_col].astype(str)

    tfidf_matrix = vectorizer.fit_transform(text_data)

    tfidf_df = pd.DataFrame(
        tfidf_matrix.toarray(),
        columns=vectorizer.get_feature_names_out()
    )

    tfidf_df.head()


## Category & Topic Insights

In [ ]:

# Top Categories

cat_cols = df.select_dtypes(include='object').columns.tolist()

for col in cat_cols[:5]:

    plt.figure(figsize=(12,5))

    df[col].value_counts().head(10).plot(kind='bar')

    plt.title(f"Top Categories - {col}")

    plt.show()


## Advanced Visualizations

In [ ]:

# Pairplot

important_numeric = df.select_dtypes(include='number').columns.tolist()[:5]

if len(important_numeric) > 1:

    sns.pairplot(
        df[important_numeric].dropna()
    )

    plt.show()


## Predictive Machine Learning Model

In [ ]:

# =========================
# MACHINE LEARNING
# =========================

numeric_cols = df.select_dtypes(include='number').columns.tolist()

target = None

# Automatically select a meaningful target
for col in numeric_cols:
    if 'citation' in col.lower() or 'score' in col.lower():
        target = col
        break

if target is None and len(numeric_cols) > 0:
    target = numeric_cols[0]

print("Selected Target:", target)

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

categorical_features = X.select_dtypes(include='object').columns.tolist()

numeric_features = [
    c for c in X.columns
    if c not in categorical_features
]

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=150,
        random_state=42,
        n_jobs=-1
    ))
])

valid_idx = y.notnull()

X = X[valid_idx]
y = y[valid_idx]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))
r2 = r2_score(y_test, preds)

print("MAE:", round(mae, 4))
print("RMSE:", round(rmse, 4))
print("R2 Score:", round(r2, 4))


## Feature Importance

In [ ]:

# Feature Importance

rf_model = model.named_steps['model']

encoded_cat = model.named_steps['preprocessor']\
    .named_transformers_['cat']\
    .named_steps['encoder']\
    .get_feature_names_out(categorical_features)

all_features = numeric_features + list(encoded_cat)

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': rf_model.feature_importances_
})

importance_df = importance_df.sort_values(
    by='Importance',
    ascending=False
)

top_features = importance_df.head(20)

plt.figure(figsize=(12,8))

sns.barplot(
    x='Importance',
    y='Feature',
    data=top_features
)

plt.title("Top 20 Important Features")

plt.show()

top_features



# Research & Business Insights

## Key Insights
- AI/ML research output has grown exponentially over time.
- Certain topics dominate publication trends.
- Paper metadata and textual signals can help estimate impact.
- Citation or popularity prediction can support recommendation systems.

## Applications
### Research Intelligence Platforms
- Trend monitoring
- Topic forecasting
- Emerging research identification

### Academic Search Engines
- Personalized recommendations
- Semantic paper search
- Citation prediction

### Enterprise AI Monitoring
- Competitive intelligence
- Innovation tracking
- Research landscape analysis

## Future Improvements
- Transformer-based NLP models
- Topic modeling (LDA/BERT)
- Citation graph analysis
- Research clustering
- Recommendation systems



# Conclusion

This notebook demonstrates a complete end-to-end AI research analytics workflow.

The project covers:
- Data preprocessing
- Advanced EDA
- NLP feature engineering
- Visualization
- Machine learning
- Feature importance analysis
- Research intelligence insights

This framework can evolve into:
- AI research intelligence systems
- Semantic academic search platforms
- Citation forecasting tools
- Automated research recommendation engines
